# 04 — Train LSTM Action Classifier
**Goal:** train a compact temporal classifier from pose sequences.  
Input shape per example is `(30, 51)`: 30 frames × 17 keypoints × (x, y, confidence).

In [ ]:
%pip install scikit-learn matplotlib

In [ ]:
from pathlib import Path
import json
import numpy as np
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

PROJECT_DIR = Path.cwd()
SEQ_DIR = PROJECT_DIR / "data" / "processed" / "sequences"
MODEL_DIR = PROJECT_DIR / "outputs" / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
X_train = np.load(SEQ_DIR / "X_train.npy")
y_train = np.load(SEQ_DIR / "y_train.npy")
X_val = np.load(SEQ_DIR / "X_val.npy")
y_val = np.load(SEQ_DIR / "y_val.npy")
label_map = json.loads((SEQ_DIR / "label_map.json").read_text(encoding="utf-8"))
NUM_CLASSES = len(label_map)

print("Device:", DEVICE)
print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Classes:", label_map)
assert len(X_train) > 0 and len(X_val) > 0, "Empty split; extract more videos or inspect split logic."

In [ ]:
# Replace NaN/inf defensively and keep values within a reasonable range.
X_train = np.nan_to_num(X_train, nan=0.0, posinf=0.0, neginf=0.0)
X_val = np.nan_to_num(X_val, nan=0.0, posinf=0.0, neginf=0.0)

train_ds = TensorDataset(torch.tensor(X_train, dtype=torch.float32),
                         torch.tensor(y_train, dtype=torch.long))
val_ds = TensorDataset(torch.tensor(X_val, dtype=torch.float32),
                       torch.tensor(y_val, dtype=torch.long))
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False)

class PoseLSTM(nn.Module):
    def __init__(self, input_size=51, hidden_size=64, num_layers=1, num_classes=5, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=num_layers,
                            batch_first=True, dropout=0.0)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, num_classes)
    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        return self.classifier(self.dropout(h_n[-1]))

model = PoseLSTM(num_classes=NUM_CLASSES).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

In [ ]:
def evaluate_loader(model, loader):
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = model(xb)
            loss_sum += criterion(logits, yb).item() * len(yb)
            correct += (logits.argmax(1) == yb).sum().item()
            total += len(yb)
    return loss_sum / max(total, 1), correct / max(total, 1)

EPOCHS = 40
best_val_loss = float("inf")
history = []

for epoch in range(1, EPOCHS+1):
    model.train()
    total_loss, total = 0.0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * len(yb)
        total += len(yb)
    train_loss = total_loss / max(total, 1)
    val_loss, val_acc = evaluate_loader(model, val_loader)
    history.append({"epoch": epoch, "train_loss": train_loss,
                    "val_loss": val_loss, "val_accuracy": val_acc})
    print(f"Epoch {epoch:02d}/{EPOCHS} | train loss {train_loss:.4f} | "
          f"val loss {val_loss:.4f} | val acc {val_acc:.3f}")
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            "model_state": model.state_dict(),
            "num_classes": NUM_CLASSES,
            "label_map": label_map,
            "input_size": 51,
            "sequence_length": X_train.shape[1],
        }, MODEL_DIR / "lstm_action_model.pt")

import pandas as pd
history_path = PROJECT_DIR / "outputs" / "reports" / "training_history.csv"
history_path.parent.mkdir(parents=True, exist_ok=True)
pd.DataFrame(history).to_csv(history_path, index=False)
print("Saved best model:", (MODEL_DIR / "lstm_action_model.pt").resolve())

In [ ]:
import matplotlib.pyplot as plt
hist = pd.DataFrame(history)
plt.figure(figsize=(8,4))
plt.plot(hist["epoch"], hist["train_loss"], label="train loss")
plt.plot(hist["epoch"], hist["val_loss"], label="validation loss")
plt.xlabel("Epoch"); plt.ylabel("Cross-entropy loss")
plt.legend(); plt.grid(True); plt.show()